In [1]:
# !pip -q install transformers faiss-cpu langchain_huggingface langchain_elasticsearch torch sentence-transformers chromadb langchain_groq

In [1]:
import faiss
import numpy as np
from elasticsearch import Elasticsearch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import os


/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from huggingface_hub import login

env_path = Path("/Users/dani/Desktop/Codebase/Keys/apiKeys.env")
if not env_path.exists():
    raise FileNotFoundError(f"Env file not found: {env_path}")

with env_path.open() as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, sep, value = line.partition("=")
        if sep != "=":
            continue
        value = value.strip()
        if (value.startswith('"') and value.endswith('"')) or (value.startswith("'") and value.endswith("'")):
            value = value[1:-1]
        os.environ.setdefault(key.strip(), value)

hf_api_token = os.environ.get("HF")
if not hf_api_token:
    raise ValueError(f"HF key not found in environment variables after loading {env_path}")

# Log in to Hugging Face
login(token=hf_api_token)

In [3]:
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast model for embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17325.58it/s]


In [27]:
from langchain_groq import ChatGroq


# Helper function to get the language model
def get_llm():
    """
    Returns the language model instance.

    This function initializes and returns a ChatGroq language model configured with the specified model name,
    temperature, maximum tokens, and other settings.

    Returns:
        ChatGroq: An instance of the ChatGroq language model.
    """
    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024,
    )
    return llm


# Elasticsearch is used here to perform text-based search on indexed documents.


In [5]:
# Use the same lightweight embedding model to avoid memory spikes and kernel crashes.
embeddings = sentence_model


In [6]:
from elasticsearch import Elasticsearch, ConnectionError
from langchain_elasticsearch import ElasticsearchStore

In [7]:
from elasticsearch import Elasticsearch

# Connect using an API key string
es = Elasticsearch(
    "https://32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io:443",
    api_key=os.environ.get("es"),
    verify_certs=False  # Set to True in production
)


/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/elasticsearch/_sync/client/__init__.py:404: SecurityWarning: Connecting to 'https://32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io:443' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [8]:
# Delete and recreate the 'movies' index
index_name = 'movies'

try:
    # Delete index if it exists
    if es.indices.exists(index=index_name):
        print(f"Deleting index '{index_name}'...")
        es.indices.delete(index=index_name)
        print(f"Index '{index_name}' deleted.")
    
    # Define the mapping for the index
    index_body = {
        "mappings": {
            "properties": {
                "content": {"type": "text"}
            }
        }
    }

# Create an index named 'movies'
    index_name = 'movies'
    if not es.indices.exists(index=index_name):
        es.indices.create(index=index_name, body=index_body)
        print(f"Index '{index_name}' created.")
    else:
        print(f"Index '{index_name}' already exists.")

    
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()


/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureR

Deleting index 'movies'...
Index 'movies' deleted.
Index 'movies' created.


In [9]:
docs = [
    {"content": "The Shawshank Redemption is a great movie."},
    {"content": "Forrest Gump is perfect for a rainy day."}
]

for i, doc in enumerate(docs, start=1):
    es.index(index=index_name, id=i, document=doc)

es.indices.refresh(index=index_name)

/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureR

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

## Reranker model

In [10]:
from sentence_transformers import SentenceTransformer

# Use a tiny sentence-transformer model for lightweight semantic reranking.
rerank_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16261.28it/s]


In [28]:
chat_groq_model = get_llm()

## Query transformation: modify and expand the query for better retrieval

In [12]:
def advanced_query_transformation(query):
    """
    Transforms the input query by adding synonyms, extensions, or modifying the structure
    for better search performance.

    Args:
        query (str): The original query.

    Returns:
        str: The transformed query with added synonyms or related terms.
    """
    # Example transformation: adding an OR clause with a related term
    expanded_query = query + " OR related_term"
    return expanded_query

# Fusion Retrieval Function
# This function retrieves documents using both vector-based and textual retrieval methods.


In [13]:
# Fusion Retrieval Function
def fusion_retrieval(query, top_k=5):
    """
    Retrieves the top_k most relevant documents using a combination of vector-based
    and textual retrieval methods.

    Args:
        query (str): The search query.
        top_k (int): The number of top documents to retrieve.

    Returns:
        list: A list of combined results from both vector and textual retrieval methods.
    """
    # Vector-based retrieval using sentence embeddings
    query_embedding = sentence_model.encode(query).tolist()
    vector_results = collection.query(query_embeddings=[query_embedding], n_results=min(top_k, len(documents)))

    # Textual retrieval using Elasticsearch
    es_body = {
        "size": top_k,  # Move size into body
        "query": {
            "match": {
                "content": query
            }
        }
    }
    es_results = es.search(index="movies", body=es_body)
    es_documents = [hit["_source"]["content"] for hit in es_results['hits']['hits']]

    # Combine results from both retrieval methods
    combined_results = vector_results['documents'][0] + es_documents

    return combined_results


## Reranking

In [14]:
import re
import numpy as np


def rerank_documents(query, documents):
    """
    Rerank documents using a lightweight sentence-transformer model.
    This keeps the notebook model-based while avoiding a heavy classification reranker.
    """
    if not documents:
        return []

    query_embedding = rerank_model.encode([query], convert_to_numpy=True)[0]
    doc_embeddings = rerank_model.encode(documents, convert_to_numpy=True)

    similarities = np.dot(doc_embeddings, query_embedding)
    ranked_indices = np.argsort(similarities)[::-1]
    return [documents[i] for i in ranked_indices]


## Summarizer

In [15]:
import torch
from transformers import pipeline

summarizer_pipeline = None


def summarizer(doc):
    """
    Generate a short summary for a document.
    Falls back to a simple extractive summary if the model cannot be loaded.
    """
    if not doc or not isinstance(doc, str):
        return ""

    text = doc.strip()
    if not text:
        return ""

    global summarizer_pipeline

    if len(text) <= 180:
        return text

    if summarizer_pipeline is None:
        try:
            summarizer_pipeline = pipeline(
                "summarization",
                model="sshleifer/distilbart-cnn-12-6",
                tokenizer="sshleifer/distilbart-cnn-12-6",
                device=-1,
            )
        except Exception:
            summarizer_pipeline = "fallback"

    if summarizer_pipeline != "fallback":
        try:
            result = summarizer_pipeline(
                text,
                max_length=80,
                min_length=20,
                do_sample=False,
            )
            return result[0]["summary_text"].strip()
        except Exception:
            pass

    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    if not sentences:
        return text[:220]

    return " ".join(sentences[:2]).strip()


In [16]:
def select_and_compress_context(documents):
    """
    Summarizes the content of the retrieved documents to create a compressed context.

    Args:
        documents (list): A list of documents to summarize.

    Returns:
        list: A list of summarized texts for each document.
    """
    summarized_context = []
    for doc in documents:
        summarized_context.append(summarizer(doc))
    return summarized_context


## Generate final answer based on all the chunks

In [17]:
def generate_answer(query, chunks, llm):
    """
    Generates an answer based on the input query and context chunks using a language model.

    Args:
        query (str): The user's query.
        chunks (list): A list of context chunks to inform the answer.
        llm (ChatGroq): An instance of the ChatGroq language model.

    Returns:
        str: The generated answer.
    """
    # Combine chunks into a single context string
    context = "\n\n".join(chunks)

    # Construct the prompt for the language model as a string
    prompt = f"""[INST]
Instruction: You're an expert in movie suggestions. Your task is to analyze carefully the context and come up with an exhaustive answer to the following question:
{query}

Here is the context to help you:

{context}

[/INST]"""

    # Invoke the language model with the prompt
    response = llm.invoke(prompt)  # Pass the prompt as a string directly

    # Since response is likely an AIMessage object, access the content directly
    generated_text = response.content

    return generated_text

## Full Advanced RAG Pipeline

In [19]:
 def advanced_rag_pipeline(query):

    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.

    Returns:
        str: The final generated answer.
    """

    # Transform and route query
    transformed_query = advanced_query_transformation(query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents)

    # Generate final answer based on the context
    final_answer = generate_answer(query, context, chat_groq_model)
    return final_answer

In [18]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")


Collection 'movies' created successfully.
Documents inserted successfully.


In [ ]:
# # Example query
# query = "What are some good movies to watch on a rainy day?"

# # Run the query through the Advanced RAG Pipeline
# answer = advanced_rag_pipeline(query)

# # Output the generated answer
# print(answer)

In [19]:
query = "What are some good movies to watch on a rainy day?"


In [20]:
transformed_query = advanced_query_transformation(query)
transformed_query

'What are some good movies to watch on a rainy day? OR related_term'

In [21]:
retrieved_documents = fusion_retrieval(transformed_query)
retrieved_documents

/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '32ec204e0b2747a5a9ab7815037614e8.us-central1.gcp.cloud.es.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['The Shawshank Redemption is a great movie to watch on a rainy day.',
 'Forrest Gump is an uplifting film perfect for a rainy afternoon.',
 'Forrest Gump is perfect for a rainy day.',
 'The Shawshank Redemption is a great movie.']

In [22]:
ranked_documents = rerank_documents(query, retrieved_documents)
ranked_documents

['The Shawshank Redemption is a great movie to watch on a rainy day.',
 'Forrest Gump is perfect for a rainy day.',
 'Forrest Gump is an uplifting film perfect for a rainy afternoon.',
 'The Shawshank Redemption is a great movie.']

In [24]:
    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents)
    context

['The Shawshank Redemption is a great movie to watch on a rainy day.',
 'Forrest Gump is perfect for a rainy day.',
 'Forrest Gump is an uplifting film perfect for a rainy afternoon.',
 'The Shawshank Redemption is a great movie.']

In [29]:
final_answer = generate_answer(query, context, chat_groq_model)
final_answer

'Based on the provided context, it seems that both "The Shawshank Redemption" and "Forrest Gump" are excellent choices for a rainy day. These movies are not only great in general but also specifically suitable for a rainy day or afternoon.\n\nConsidering this, here are some more movie suggestions that might be perfect for a rainy day:\n\n**Dramas:**\n\n1. **The Shawshank Redemption** (1994) - A highly acclaimed drama about hope, redemption, and friendship.\n2. **Forrest Gump** (1994) - A heartwarming and uplifting film about a man\'s journey through life, love, and friendship.\n3. **The Pursuit of Happyness** (2006) - A biographical drama about a struggling single father\'s journey to build a better life for himself and his son.\n4. **The Notebook** (2004) - A romantic drama about the love story of two young souls from different social classes.\n5. **Schindler\'s List** (1993) - A historical drama about the true story of Oskar Schindler, a German businessman who saves the lives of thou